## ***Import Libraries***

In [1]:
import numpy as np
import pandas as pd
import re
from typing import List, Dict
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown

## ***Load and Preview Resume & Job Description Datasets***


In [2]:
# Load the resume CSV
resume_df = pd.read_csv("resume.csv")

# Load the job descriptions Excel
jobs_df = pd.read_excel("all_jobs.xlsx")

# Display first 10 rows from each dataset
print("📄 First 10 Resumes:")
display(resume_df[['ID', 'Resume_str', 'Category']].head(10))

print("💼 First 10 Job Descriptions:")
display(jobs_df[['id', 'title', 'cleaned_description']].head(10))


📄 First 10 Resumes:


,ID,Resume_str,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...",HR
2,33176873,HR DIRECTOR Summary Over 2...,HR
3,27018550,HR SPECIALIST Summary Dedica...,HR
4,17812897,HR MANAGER Skill Highlights ...,HR
5,11592605,HR GENERALIST Summary Dedic...,HR
6,25824789,HR MANAGER Summary HUMAN RES...,HR
7,15375009,HR MANAGER Professional Summa...,HR
8,11847784,HR SPECIALIST Summary Posses...,HR
9,32896934,HR CLERK Summary Translates ...,HR


💼 First 10 Job Descriptions:


,id,title,cleaned_description
0,487e95701c0d55b3,Cybersecurity Operations Senior Consultant,riscpoint is seeking a conceptual thinker with...
1,679ff0528b73610f,Senior Information Security Engineer (GRC),company description marketaxess is on a journe...
2,41f6f886cd9b9d1d,Information System Security Officer II,"global resource solutions, inc. (grs) is seeki..."
3,f82f829d7cf62384,Cyber Operations Support Specialist,**company overview:** by light professional it...
4,895d9f28e9e5ee10,"Identity & Access Management, Analyst",**do you want your voice heard and your action...
5,dae2a424557516af,Cyber Security Consultant,**overview** we are seeking a knowledgeable an...
6,cffabc1716f19590,Identity and Access Management,**role: identity and access management sailpoi...
7,f122dcbb01c96ec5,Cyber Security Government Consultant,**job summary** we are seeking a skilled and m...
8,3b2ce16e85a71826,Senior Security Consultant,**senior security consultan****t** **location:...
9,7f0863af245ceb16,Sr. Cloud Data Security Engineer,**the role:** as a sr. cloud architect you wil...


## ***Clean Resume and Job Text by Removing Special Characters and Extra Spaces***


In [3]:
def clean_text(text):
    text = str(text).lower()                       # Lowercase
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)     # Remove special chars
    text = re.sub(r'\s+', ' ', text).strip()       # Remove extra spaces
    return text

resume_df['cleaned_resume'] = resume_df['Resume_str'].apply(clean_text)
jobs_df['cleaned_job'] = jobs_df['cleaned_description'].apply(clean_text)

print("resume_df columns ➤", resume_df.columns.tolist())
print("jobs_df columns ➤", jobs_df.columns.tolist())


resume_df columns ➤ ['ID', 'Resume_str', 'Resume_html', 'Category', 'cleaned_resume']
jobs_df columns ➤ ['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0', 'id', 'site', 'job_url', 'job_url_direct', 'title', 'company', 'location', 'job_type', 'date_posted', 'salary_source', 'interval', 'min_amount', 'max_amount', 'currency', 'is_remote', 'job_level', 'job_function', 'company_industry', 'listing_type', 'emails', 'description', 'company_url', 'company_url_direct', 'company_addresses', 'company_num_employees', 'company_revenue', 'company_description', 'logo_photo_url', 'banner_photo_url', 'ceo_name', 'ceo_photo_url', 'mean_salary', 'cleaned_description', 'cleaned_job']


## ***Extract Relevant Skills from Resumes and Job Descriptions***


In [4]:
# Define a basic skills list (you can expand this later)
skills = ['excel', 'python', 'management', 'security', 'recruiting', 'cloud', 'aws', 'linux', 'communication']

# Function to extract matched skills from a string
def extract_skills(text):
    return [skill for skill in skills if skill in text.lower()]

# Extract skills from resumes and jobs
resume_df['skills'] = resume_df['cleaned_resume'].apply(extract_skills)
jobs_df['skills'] = jobs_df['cleaned_job'].apply(extract_skills)


## ***Compare Resume and Job Skills to Identify Skill Gaps***


In [5]:
def compare_skills(resume_skills, job_skills):
    return list(set(job_skills) - set(resume_skills))  # Skills missing in resume

# Example: Compare first resume with first job
resume_skills = resume_df.loc[0, 'skills']
job_skills = jobs_df.loc[0, 'skills']

print("Resume Skills:", resume_skills)
print("Job Skills:", job_skills)
print("Skill Gap:", compare_skills(resume_skills, job_skills))


Resume Skills: ['management', 'aws']
Job Skills: ['excel', 'management', 'security', 'cloud', 'aws', 'communication']
Skill Gap: ['security', 'cloud', 'excel', 'communication']


## ***Resume Classification Using TF-IDF and Logistic Regression***


In [6]:
# TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(resume_df['cleaned_resume'])
y = resume_df['Category']

# Split and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = LogisticRegression()
clf.fit(X_train, y_train)

# Evaluation
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.6458752515090543
[[25  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  3  0  1  0  0  0  0  0]
 [ 0 17  0  0  1  0  0  0  0  0  0  0  1  0  0  0  0  0  8  1  0  1  1  0]
 [ 1  0  1  0  0  0  0  0  0  1  0  0  0  0  0  1  0  0  0  0  1  0  1  2]
 [ 0  1  0  7  3  0  0  1  0  0  0  0  0  1  0  2  0  0  0  0  0  0  5  0]
 [ 0  1  0  1  2  0  0  0  0  0  1  0  1  1  0  0  1  0  0  1  3  1  1  4]
 [ 0  3  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  1  0]
 [ 0  0  0  0  0  0 18  0  0  0  0  0  0  0  0  2  0  0  0  0  1  0  0  0]
 [ 1  0  1  0  1  0  0 16  0  0  0  0  1  0  0  0  2  0  0  0  1  0  0  0]
 [ 0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0]
 [ 0  0  0  0  0  0  0  1  0 17  0  0  0  0  0  0  1  0  1  1  1  2  3  0]
 [ 0  1  0  1  1  0  2  0  0  0 17  0  0  0  0  0  0  0  2  0  0  0  0  0]
 [ 0  0  0  2  1  0  1  0  0  0  0 25  0  0  0  2  0  1  1  0  0  1  0  0]
 [ 0  1  0  0  0  0  0  2  0  0  0  1  5  0  0  1  0  0  2  1  5  2  0 

c:\Users\pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

## ***Train a Neural Network for Resume Classification***


In [7]:
# Encode categories
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_cat = to_categorical(y_encoded)

# Use same X from TF-IDF above
X_train, X_test, y_train, y_test = train_test_split(X.toarray(), y_cat, test_size=0.2)

model = Sequential([
    Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(y_cat.shape[1], activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=5, batch_size=32, validation_split=0.2)

c:\Users\pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 97ms/step - accuracy: 0.1401 - loss: 3.1099 - val_accuracy: 0.4070 - val_loss: 2.6468
Epoch 2/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.4594 - loss: 2.3125 - val_accuracy: 0.5578 - val_loss: 1.8191
Epoch 3/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.6076 - loss: 1.5049 - val_accuracy: 0.5578 - val_loss: 1.5544
Epoch 4/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.6702 - loss: 1.2153 - val_accuracy: 0.5905 - val_loss: 1.4398
Epoch 5/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.7457 - loss: 0.8928 - val_accuracy: 0.5578 - val_loss: 1.4106


## ***Job Recommendation for a Resume Using Cosine Similarity***


In [8]:
# Combine title + job desc for job profile
jobs_df['job_profile'] = jobs_df['title'] + " " + jobs_df['cleaned_job']

# TF-IDF vectorization
job_vectors = vectorizer.fit_transform(jobs_df['job_profile'])
resume_vectors = vectorizer.transform(resume_df['cleaned_resume'])

# Example: Recommend jobs for resume[0]
similarities = cosine_similarity(resume_vectors[0], job_vectors)
top_indices = similarities[0].argsort()[-5:][::-1]  # Top 5 jobs

display(Markdown("### Resume 0 Content"))
display(Markdown(resume_df['cleaned_resume'][0]))

print("Top job matches for Resume 0:")
display(jobs_df[['title', 'cleaned_description']].iloc[top_indices])

### Resume 0 Content

hr administratormarketing associate hr administrator summary dedicated customer service manager with 15 years of experience in hospitality and customer service management respected builder and leader of customerfocused teams strives to instill a shared enthusiastic commitment to customer service highlights focused on customer satisfaction team management marketing savvy conflict resolution techniques training and development skilled multitasker client relations specialist accomplishments missouri dot supervisor training certification certified by ihg in customer loyalty and marketing by segment hilton worldwide general manager training certification accomplished trainer for cross server hospitality systems such as hilton onq micros opera pms fidelio opera reservation system ors holidex completed courses and seminars in customer service sales strategies inventory control loss prevention safety time management leadership and performance assessment experience hr administratormarketing associate hr administrator dec 2013 to current company name city state helps to develop policies directs and coordinates activities such as employment compensation labor relations benefits training and employee services prepares employee separation notices and related documentation keeps records of benefits plans participation such as insurance and pension plan personnel transactions such as hires promotions transfers performance reviews and terminations and employee statistics for government reporting advises management in appropriate resolution of employee relations issues administers benefits programs such as life health dental insurance pension plans vacation sick leave leave of absence and employee assistance marketing associate designed and created marketing collateral for sales meetings trade shows and company executives managed the inhouse advertising program consisting of print and media collateral pieces assisted in the complete design and launch of the companys website in 2 months created an official company page on facebook to facilitate interaction with customers analyzed ratings and programming features of competitors to evaluate the effectiveness of marketing strategies advanced medical claims analyst mar 2012 to dec 2013 company name city state reviewed medical bills for the accuracy of the treatments tests and hospital stays prior to sanctioning the claims trained to interpret the codes icd9 cpt and terminology commonly used in medical billing to fully understand the paperwork that is submitted by healthcare providers required to have organizational and analytical skills as well as computer skills knowledge of medical terminology and procedures statistics billing standards data analysis and laws regarding medical billing assistant general manager jun 2010 to dec 2010 company name city state performed duties including but not limited to budgeting and financial management accounting human resources payroll and purchasing established and maintained close working relationships with all departments of the hotel to ensure maximum operation productivity morale and guest service handled daily operations and reported directly to the corporate office hired and trained staff on overall objectives and goals with an emphasis on high customer service marketing and advertising working on public relations with the media government and local businesses and chamber of commerce executive support marketing assistant jul 2007 to jun 2010 company name city state provided assistance to various department heads executive marketing customer service human resources managed frontend operations to ensure friendly and efficient transactions ensured the swift resolution of customer issues to preserve customer loyalty while complying with company policies exemplified the secondtonone customer service delivery in all interactions with customers and potential clients reservation front office manager jun 2004 to jul 2007 company name city state owner partner dec 2001 to may 2004 company name city state price integrity coordinator aug 1999 to dec 2001 company name city state education na business administration 1999 jefferson college city state business administration marketing advertising high school diploma college prep studies 1998 sainte genevieve senior high city state awarded american shrubel leadership scholarship to jefferson college skills accounting ads advertising analytical skills benefits billing budgeting clients customer service data analysis delivery documentation employee relations financial management government relations human resources insurance labor relations layout marketing marketing collateral medical billing medical terminology office organizational payroll performance reviews personnel policies posters presentations public relations purchasing reporting statistics website

Top job matches for Resume 0:


,title,cleaned_description
36959,Payroll Specialist,: the position the city of santa rosa is hirin...
52493,HUMAN RESOURCES MANAGER,"**salary :** $143,282.04 - $174,160.20 annuall..."
57619,Field Marketing Manager - Central,**redefine the future of customer experiences....
52639,Human Resources Operations Manager (Department...,"**salary :** $111,106.53 - $177,770.53 annuall..."
58927,Marketing Associate,job responsibilities **about greystar** greyst...


## ***Making files for webapp***


In [9]:
import joblib

# Save trained logistic regression model
joblib.dump(clf, 'resume_classifier_model.pkl')

# Save TF-IDF vectorizer used to transform resumes
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')


['tfidf_vectorizer.pkl']